# 08 - pAgo QC evidence inventory

This notebook calls the pAgo QC evidence inventory snapshot module. The evidence rules live in `src/pago_pipeline/pago_qc.py`; artifact writing lives in `src/pago_pipeline/pago_qc_snapshot.py`.

In [1]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pago_qc_snapshot as pago_qc_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pago_qc_snapshot_module = importlib.reload(pago_qc_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_pago_qc_evidence_inventory = (
    pago_qc_snapshot_module.resolve_pago_qc_evidence_inventory
)

In [2]:
# =============================================================================
# CELL 2 - Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 - Define input and output paths
# =============================================================================

METADATA_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "02-intermediate"
    / "protein_metadata_csv"
    / "latest"
    / "protein_metadata.csv"
)

SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "03-features"
    / "pago_qc"
    / "evidence_inventory"
)

print(f"Metadata CSV: {METADATA_CSV_PATH}")
print(f"Snapshot root directory: {SNAPSHOT_ROOT_DIRECTORY}")

Metadata CSV: C:\Programming\Python\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\protein_metadata.csv
Snapshot root directory: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory


In [4]:
# =============================================================================
# CELL 4 - Run evidence inventory
# =============================================================================

inventory_payload = resolve_pago_qc_evidence_inventory(
    snapshot_mode=SnapshotMode.reuse_latest_or_create,
    metadata_csv_file_path=METADATA_CSV_PATH,
    snapshot_root_directory=SNAPSHOT_ROOT_DIRECTORY,
)

inventory_manifest = inventory_payload["manifest"]

print(f"Metadata rows: {inventory_manifest['metadata_row_count']:,}")
print(f"Snapshot directory: {inventory_payload['snapshot_directory']}")
print(f"Evidence flags: {inventory_payload['evidence_flags_file_path']}")
print(f"Evidence counts: {inventory_payload['evidence_counts_file_path']}")
print(f"Labelled records: {inventory_payload['labelled_records_file_path']}")
print(f"Label counts: {inventory_payload['label_counts_file_path']}")
print(
    f"Filter decision counts: "
    f"{inventory_payload['filter_decision_counts_file_path']}"
)
print(f"Manifest: {inventory_payload['manifest_file_path']}")

Metadata rows: 1,010
Snapshot directory: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446
Evidence flags: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446\evidence_flags.csv
Evidence counts: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446\evidence_counts.csv
Labelled records: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446\labelled_records.csv
Label counts: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446\label_counts.csv
Filter decision counts: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\snapshots\2026-06-02T11-47-47Z__q_ad6b1adf4446\filter_decision_counts.csv
Manifest: 

In [5]:
# =============================================================================
# CELL 5 - Inspect first-pass evidence counts
# =============================================================================

evidence_counts_df = inventory_payload["evidence_counts"]
evidence_counts_df.sort_values("count", ascending=False)

,flag,count,fraction
0,has_classic_piwi_text_anywhere,973,0.963366
2,has_any_piwi_related_text_anywhere,973,0.963366
3,has_piwi_text_anywhere,973,0.963366
5,has_any_piwi_related_evidence,973,0.963366
6,has_piwi_region,865,0.856436
4,has_any_piwi_related_region,865,0.856436
7,has_classic_piwi_region,865,0.856436
16,has_cdd_region,865,0.856436
15,has_active_site_annotation,620,0.613861
14,has_active_or_functional_site_annotation,620,0.613861


In [6]:
# =============================================================================
# CELL 6 - Inspect label counts
# =============================================================================

label_counts_df = inventory_payload["label_counts"]
label_counts_df.sort_values("count", ascending=False)

,primary_label,count,fraction
0,classic_piwi_candidate,973,0.963366
1,low_evidence_or_unrelated,37,0.036634


In [7]:
# =============================================================================
# CELL 7 - Inspect filter decision counts
# =============================================================================

filter_decision_counts_df = inventory_payload["filter_decision_counts"]
filter_decision_counts_df.sort_values("count", ascending=False)

,qc_decision,count,fraction
0,review,581,0.575248
1,include,383,0.379208
2,exclude,46,0.045545
